# Método analítico para $\alpha$ (Line search)

Asumiendo que $J$ es convexa (paraboloide) podemos encontrar el valor óptimo de $\alpha$  definiendo la siguente función auxiliar

$$
\varphi(\alpha) = J[\theta_k - \alpha \nabla J(\theta_k)].
$$

Entonces podemos usar la regla de la cadena para encontrar el mínimo para $\varphi$. De este modo minimizamos haciendo

$$
\varphi'(\alpha) = \nabla J(\theta_k - \alpha \nabla J(\theta_k)) \cdot \nabla J(\theta_k) = 0,
$$

lo que es

$$
\varphi'(\alpha) = \nabla J(\theta_{k+1}) \cdot \nabla J(\theta_k) = 0,
$$

por lo que $\nabla J(\theta_{k+1})$ es ortogonal a $\nabla J(\theta_k)$.

Multiplicando la expresión $\theta_{k+1} = \theta_k - \alpha \nabla J(\theta_k)$ por $A$ y restandole $b$, tenemos

$$
A \theta_{k+1} - b = A \theta_k - b - \alpha A \nabla J(\theta_k)
$$

y definiendo los residuales 

$$
\begin{aligned}
r_k &= A \theta_k - b =  \nabla J(\theta_k) \\
r_{k+1} &= A \theta_{k+1} - b = \nabla J(\theta_{k+1}),
\end{aligned}
$$

podemos escribir

$$
r_{k+1} = r_k - \alpha A \nabla J(\theta_k) = r_k - \alpha A r_k.
$$

Note que $r_{k+1} \cdot r_k = 0$ (son ortogonales). Podemos pre-multiplicar la expresión anterior por $r_k^T$ para obtener:

$$
\begin{aligned}
r_k^T r_{k+1}  &= r_k^T (r_k - \alpha A r_k) \\
r_{k+1} \cdot r_k &= r_k^T r_k - \alpha r_k^T A r_k \\
0 &= \|r_k\|^2 - \alpha r_k^T A r_k
\end{aligned}
$$

Y resolviendo para $\alpha$ tenemos

$$
\alpha = \frac{\|r_k\|^2}{r_k^T A r_k}
$$

Tenemos entonces una expesión analitica para el parámetro $\alpha$ del algoritmo "line search". Finalmente, necesitamos definir un número máximo de iteraciones posibles y establecer la tolerancia de nuestra estimación. Una forma de calcular el error es usar la norma del vector $r_k$. Continuamos con la iteración siempre que esta norma sea mayor que la tolerancia. 

A continuación aplicamos este algoritmo al ejemplo propuesto en el cuadernillo anterior

**Ejemplo**

Considere la siguente matriz y vector

$$
A = X^T X = 
\begin{pmatrix}
10 & 0 \\
0 & 1
\end{pmatrix}, \quad b = X^T y
\begin{pmatrix}
1 \\
-1
\end{pmatrix}
$$

Con los cuales contruimos la función objetivo (paraboloide)

$$
J(\theta) = \frac{1}{2}\theta^T A\theta - b^T \theta  = \frac{1}{2}\theta^T X^T X\theta - X^T y \theta = 5 \theta_0^2 + \frac{1}{2} \theta_1^2 - \theta_0 + \theta_1
$$

Las siguentes celdas de código demuestran el proceso para obtener una aproximación a un punto de mínima usando el método de descenso del gradiente con $\alpha$ en funcion de $r_k$ y $A$.

In [1]:
# requirements
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d
import pandas as pd

plt.style.use('ggplot')
%matplotlib ipympl

In [2]:
def steepestDescent(A, b, theta, tol, M):
    """
    Steepest descent algorithm with analytic alpha.
    
    This algorithm takes as input A = X.T @ X and
    b = X.T @ y.
    
    This is not a practical implementation as it
    returns the recors of J and theta parameter to
    plot the way down to the minimum.
    
    Input parameters
    A: defined as X.T @ X
    b: defined as X.T @ y
    theta: initial approximation
    tol: tolerance
    M: maximum number of iterations
    
    Output
    theta_rec: recorded parameters theta
    J_rec: recorded values of J at theta_rec
    """
    
    J = 0.5*theta.T @ A @ theta - b.T @ theta
    theta_rec = theta.T
    J_rec = J
    k = 0
    while k < M:
        rk = A @ theta - b # Gradient
        if np.linalg.norm(rk) < tol:
            print('The procedure was successful.')
            return theta_rec, J_rec
        
        k += 1
        alpha = np.linalg.norm(rk)**2/(rk.T @ A @ rk)
        theta = theta - alpha*rk
        J = 0.5*theta.T @ A @ theta - b.T @ theta
        theta_rec = np.vstack((theta_rec, theta.T)) # Store theta_k
        J_rec = np.vstack((J_rec, J)) # Store J(theta_k)
        
    print("Maximum number of iterations reached")
    return theta_rec, J_rec

Probemos el algoritmo

In [3]:
A = np.array([[10, 0], [0, 1]])
b = np.array([[1], [-1]])

theta = np.array([[1], [1]])
tol = 0.01
M = 1000

theta_rec, J_rec = steepestDescent(A, b, theta, tol, M)
data = np.hstack((theta_rec, J_rec))
pd.DataFrame(data, columns=["theta0", "theta1", "J(theta)"])

The procedure was successful.


,theta0,theta1,J(theta)
0,1.000000,1.000000,5.500000
1,0.060197,0.791155,1.062039
2,0.339808,-0.467094,-0.120468
3,0.089394,-0.522742,-0.435550
4,0.163897,-0.858006,-0.519504
5,0.097174,-0.872833,-0.541874
6,0.117026,-0.962165,-0.547835
7,0.099247,-0.966116,-0.549423
8,0.104537,-0.989919,-0.549846
9,0.099799,-0.990972,-0.549959


In [4]:
# Cost function
def cost(theta0, theta1):
    return 5*theta0**2 + 0.5*theta1**2 - theta0 + theta1

theta0 = np.linspace(-1, 1, 100)
theta1 = np.linspace(-3, 1, 100)
[T0, T1] = np.meshgrid(theta0, theta1)

J = cost(T0, T1)

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(1,1,1, projection="3d")

ax.plot_surface(T0, T1, J, cmap ="jet", alpha=0.5)
ax.contour(T0, T1, J, zdir='z', cmap ="jet", offset = -5)

ax.plot(theta_rec[:, 0], theta_rec[:, 1], J_rec[:, 0], "r")
ax.plot(theta_rec[:, 0], theta_rec[:, 1], -5, "r")

ax.set_zlim3d(-5, 10)
ax.set_xlabel(r"$\theta_0$")
ax.set_ylabel(r"$\theta_1$")
ax.set_zlabel(r"$J(\theta_0, \theta_1)$")
plt.tight_layout()
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

La ruta de descenso se muestra en rojo. El camino zigzaguea con giros de 90 grados hasta converger a un punto muy cercano al mínimo analítico dado por $\theta = (0.1, -1)^T$. El método de Descenso más pronunciado converge para funciones convexas. Sin embargo, el comportamiento en zigzag de la ruta de búsqueda podría ser costoso. El comportamiento de zigzag puede ser mucho peor si la dimensionalidad del problema es alta.

El caso de una superficie parabólica cuadrática es un caso especial en el que podemos encontrar $\alpha$ analíticamente. En la práctica. Sin embargo, las funciones de coste no son necesariamente cuadráticas, por lo que debemos cambiar la estrategia para encontrar $\alpha$ y controlar la longitud de los pasos en zigzag.